# SEC Financial Data Pipeline

Pipeline en **2 étapes** :
1. **`SEC_RAW/`** — Téléchargement & extraction des métriques SEC (une seule passe API)
2. **`SEC_FINAL/`** — Calcul des ratios de croissance + ajout des prix Yahoo Finance

---

## 0. Configuration globale

In [1]:
import requests
import pandas as pd
import numpy as np
import os
import time
import yfinance as yf

# ──────────────────────────────────────────
# SEC API
# ──────────────────────────────────────────
HEADERS = {"User-Agent": "Jean Lassae ralic29240@isfew.com"}

# ──────────────────────────────────────────
# DOSSIERS DE SORTIE
# ──────────────────────────────────────────
RAW_DIR   = "./SEC_RAW"    # Étape 1 : données brutes pivotées
FINAL_DIR = "./SEC_FINAL"  # Étape 2 : croissance + prix

os.makedirs(RAW_DIR,   exist_ok=True)
os.makedirs(FINAL_DIR, exist_ok=True)

# ──────────────────────────────────────────
# MÉTRIQUES À EXTRAIRE 
# ──────────────────────────────────────────
TARGET_METRICS = {
    "Assets",
    "LiabilitiesAndStockholdersEquity",
    "IncomeTaxExpenseBenefit",
    "CashAndCashEquivalentsAtCarryingValue",
    "RetainedEarningsAccumulatedDeficit",
    "StockholdersEquity",
    "AccumulatedOtherComprehensiveIncomeLossNetOfTax",
    "NetIncomeLoss",
    "NetCashProvidedByUsedInFinancingActivities",
    "NetCashProvidedByUsedInInvestingActivities",
    "NetCashProvidedByUsedInOperatingActivities",
    "AssetsCurrent",
    "LiabilitiesCurrent",
    "ComprehensiveIncomeNetOfTax",
    "PropertyPlantAndEquipmentNet",
    "OperatingIncomeLoss",
    "ShareBasedCompensation",
    "OtherAssetsNoncurrent",
    "CommonStockValue",
    "Goodwill",
    "OtherLiabilitiesNoncurrent",
    "Liabilities",
    "PaymentsToAcquirePropertyPlantAndEquipment",
    "AccountsPayableCurrent",
    "InterestExpense",
    "ProfitLoss",
    "AccountsReceivableNetCurrent",
    "StockholdersEquityIncludingPortionAttributableToNoncontrollingInterest",
    "OtherNonoperatingIncomeExpense",
    "InventoryNet",
    "DepreciationDepletionAndAmortization",
    "IncreaseDecreaseInInventories",
    "IncreaseDecreaseInAccountsReceivable",
    "MinorityInterest",
    "IntangibleAssetsNetExcludingGoodwill",
    "SellingGeneralAndAdministrativeExpense",
    "AdditionalPaidInCapitalCommonStock",
}

print(f"Config chargée — {len(TARGET_METRICS)} métriques ciblées")

Config chargée — 37 métriques ciblées


## 1. Chargement du fichier tickers

In [2]:
df_tickers = pd.read_csv("tickers.csv", index_col=0)
df_tickers.drop(columns=["Industry"], inplace=True, errors="ignore")
df_tickers.dropna(inplace=True)

# Garder uniquement Nasdaq et NYSE
df_tickers = df_tickers[df_tickers["exchange"].isin(["Nasdaq", "NYSE"])]

# Limiter à 500 entreprises par secteur
df_tickers = df_tickers.groupby("Sector").head(500)

print("Répartition par secteur :")
print(df_tickers["Sector"].value_counts())
print(f"\nTotal : {len(df_tickers)} entreprises")
df_tickers.head()

Répartition par secteur :
Sector
Technology                500
Consumer Cyclical         500
Financial Services        500
Healthcare                500
Industrials               500
Real Estate               405
Communication Services    280
Basic Materials           280
Energy                    261
Consumer Defensive        254
Utilities                 126
Name: count, dtype: int64

Total : 4106 entreprises


,cik,name,ticker,exchange,Sector
0,1045810,NVIDIA CORP,NVDA,Nasdaq,Technology
1,1652044,Alphabet Inc.,GOOGL,Nasdaq,Communication Services
2,320193,Apple Inc.,AAPL,Nasdaq,Technology
3,789019,MICROSOFT CORP,MSFT,Nasdaq,Technology
4,1018724,AMAZON COM INC,AMZN,Nasdaq,Consumer Cyclical


## Étape 1 - Téléchargement SEC & extraction des métriques

> **Sortie :** `SEC_RAW/<Sector>/<TICKER>.csv`  
> Une ligne par filing (date), une colonne par métrique.

In [ ]:
def fetch_company_facts(cik: str) -> dict | None:
    """Appel API SEC — retourne le JSON ou None en cas d'erreur."""
    url = f"https://data.sec.gov/api/xbrl/companyfacts/CIK{cik}.json"
    resp = requests.get(url, headers=HEADERS)
    if resp.status_code != 200:
        return None
    return resp.json()


def extract_and_save(cik: str, ticker: str, sector: str, nodata: list) -> None:
    """
    1. Appel API SEC (une seule fois).
    2. Filtre sur TARGET_METRICS + unité USD + forms 10-K/10-Q.
    3. Pivot → une colonne par métrique.
    4. Sauvegarde dans SEC_RAW/<sector>/<ticker>.csv
    """
    print(f"  ⬇  {ticker} ({sector})")

    data = fetch_company_facts(str(cik).zfill(10))
    if data is None:
        nodata.append(ticker)
        return

    facts = data.get("facts", {}).get("us-gaap", {})
    rows = []

    for metric, metric_data in facts.items():

        # ── filtre strict sur les métriques souhaitées ──
        if metric not in TARGET_METRICS:
            continue

        for unit, values in metric_data.get("units", {}).items():
            if unit != "USD":
                continue

            for entry in values:
                if entry.get("form") not in ("10-Q", "10-K"):
                    continue
                if entry.get("filed") is None or entry.get("val") is None:
                    continue

                rows.append({
                    "filed":  entry["filed"],
                    "metric": metric,
                    "value":  entry["val"],
                    "form":   entry["form"],
                    "fp":     entry.get("fp"),
                    "fy":     entry.get("fy"),
                })

    if not rows:
        nodata.append(ticker)
        return

    df = pd.DataFrame(rows)
    df["filed"] = pd.to_datetime(df["filed"])

    # Dédoublonnage : une valeur par (metric, filed)
    df = (
        df.sort_values("filed", ascending=False)
          .drop_duplicates(subset=["metric", "filed"], keep="first")
    )

    # Enrichissement
    df["year"]   = df["fy"]
    df["period"] = df.apply(lambda r: r["fp"] if r["form"] == "10-Q" else "K10", axis=1)
    df["type"]   = df["form"].map({"10-Q": "Q", "10-K": "K"})

    # Pivot
    df_pivot = (
        df.pivot_table(
            index=["filed", "year", "period", "type"],
            columns="metric",
            values="value",
            aggfunc="first",
        )
        .reset_index()
        .sort_values("filed")
    )
    df_pivot.columns.name = None  # enlève le label "metric" sur les colonnes

    # Sauvegarde
    out_dir = os.path.join(RAW_DIR, sector)
    os.makedirs(out_dir, exist_ok=True)
    df_pivot.to_csv(os.path.join(out_dir, f"{ticker}.csv"), index=False)
    print(f" {ticker} — {len(df_pivot)} lignes, {df_pivot.shape[1]-4} métriques")


# ──────────────────────────────────────────
# BOUCLE PRINCIPALE
# ──────────────────────────────────────────

nodata = []

for _, row in df_tickers.iterrows():
    if pd.isna(row["cik"]) or pd.isna(row["ticker"]):
        continue
    extract_and_save(row["cik"], row["ticker"], row["Sector"], nodata)
    time.sleep(0.2)   # respect du rate-limit SEC

# Résumé
print(f"\n Terminé — {len(nodata)} ticker(s) sans données :")
print(nodata)

In [4]:
# Compte CSV par secteur dans SEC_RAW
for folder in sorted(os.listdir(RAW_DIR)):
    path = os.path.join(RAW_DIR, folder)
    if os.path.isdir(path):
        n = len([f for f in os.listdir(path) if f.endswith(".csv")])
        print(f"  {folder:<30} {n} fichiers")
#print(f"  {'TOTAL':<30} {len(df_tickers) - len(nodata)} fichiers")

  Basic Materials                158 fichiers
  Communication Services         188 fichiers
  Consumer Cyclical              392 fichiers
  Consumer Defensive             184 fichiers
  Energy                         175 fichiers
  Financial Services             355 fichiers
  Healthcare                     442 fichiers
  Industrials                    423 fichiers
  Real Estate                    388 fichiers
  Technology                     404 fichiers
  Utilities                      91 fichiers


## Étape 2 - Calcul des croissances + ajout des prix Yahoo Finance

> **Entrée  :** `SEC_RAW/<Sector>/<TICKER>.csv`  
> **Sortie  :** `SEC_FINAL/<Sector>/<TICKER>.csv`  
> Toutes les métriques disponibles sont traitées (aucune sélection manuelle).

In [ ]:
# ──────────────────────────────────────────
# HELPERS
# ──────────────────────────────────────────

META_COLS = {"filed", "year", "period", "type"}


def safe_growth(curr, prev):
    """Croissance YoY sécurisée — retourne NaN si impossible."""
    if pd.isna(curr) or pd.isna(prev) or prev == 0:
        return np.nan
    return (curr - prev) / abs(prev)


def download_prices(ticker: str) -> pd.DataFrame | None:
    """Télécharge l'historique de prix Yahoo Finance (une seule fois)."""
    try:
        stock = yf.Ticker(ticker)
        df = stock.history(start="2000-01-01", auto_adjust=True)

        if df is None or df.empty:
            print(f"    ⚠️  Pas de prix pour {ticker}")
            return None

        df = df.reset_index()
        df["Date"] = pd.to_datetime(df["Date"]).dt.tz_localize(None)
        df = df[["Date", "Close"]]
        return df

    except Exception as e:
        print(f"    ❌ Erreur prix {ticker}: {e}")
        return None


def nearest_price_before(df_price: pd.DataFrame, date: pd.Timestamp, lookback: int = 7) -> float | None:
    """
    Prix de clôture le plus récent AVANT la date du filing (j-1 ou antérieur).
    Cherche dans une fenêtre de `lookback` jours calendaires vers le passé.
    Garantit qu'aucun prix du jour J ou après n'est retourné.
    """
    window = df_price[
        (df_price["Date"] >= date - pd.Timedelta(days=lookback)) &
        (df_price["Date"] < date)          # strict : exclut le jour J
    ]
    if window.empty:
        return None
    return float(window.iloc[-1]["Close"]) # le plus récent avant J


def nearest_price_after(df_price: pd.DataFrame, date: pd.Timestamp, days: int = 7) -> float | None:
    """
    Prix de clôture le plus proche APRÈS (ou égal à) la date cible.
    Cherche dans une fenêtre de `days` jours calendaires vers le futur.
    """
    window = df_price[
        (df_price["Date"] >= date) &
        (df_price["Date"] <= date + pd.Timedelta(days=days))
    ]
    if window.empty:
        return None
    return float(window.iloc[0]["Close"])


# ──────────────────────────────────────────
# RATIOS DÉFINIS (numérateur, dénominateur)
# ──────────────────────────────────────────

RATIOS = {
    "current_ratio":         ("AssetsCurrent",                               "LiabilitiesCurrent"),
    "debt_to_equity":        ("Liabilities",                                 "StockholdersEquity"),
    "roa":                   ("NetIncomeLoss",                               "Assets"),
    "roe":                   ("NetIncomeLoss",                               "StockholdersEquity"),
    "cash_ratio":            ("CashAndCashEquivalentsAtCarryingValue",       "LiabilitiesCurrent"),
    "operating_margin":      ("OperatingIncomeLoss",                         "Assets"),
    "capex_to_cfo":          ("PaymentsToAcquirePropertyPlantAndEquipment",  "NetCashProvidedByUsedInOperatingActivities"),
    "goodwill_to_assets":    ("Goodwill",                                    "Assets"),
    "retained_to_equity":    ("RetainedEarningsAccumulatedDeficit",          "StockholdersEquity"),
    "interest_coverage":     ("OperatingIncomeLoss",                         "InterestExpense"),
    "sga_to_assets":         ("SellingGeneralAndAdministrativeExpense",      "Assets"),
    "nwc_ratio":             ("AssetsCurrent",                               "LiabilitiesCurrent"),
    "cfo_to_liabilities":    ("NetCashProvidedByUsedInOperatingActivities",  "Liabilities"),
    "intangibles_to_assets": ("IntangibleAssetsNetExcludingGoodwill",        "Assets"),
    "equity_ratio":          ("StockholdersEquity",                          "Assets"),
}

# Horizons de prix futurs à calculer
PRICE_HORIZONS = [
    ("1d",  pd.Timedelta(days=1)),
    ("1w",  pd.Timedelta(weeks=1)),
    ("2w",  pd.Timedelta(weeks=2)),
    ("1m",  pd.DateOffset(months=1)),
    ("2m",  pd.DateOffset(months=2)),
    ("3m",  pd.DateOffset(months=3)),
]


def safe_ratio(num, den):
    """Division sécurisée — retourne NaN si impossible."""
    try:
        if pd.isna(num) or pd.isna(den) or den == 0:
            return np.nan
        return num / den
    except Exception:
        return np.nan


def compute_growth_and_prices(input_path: str, output_path: str, ticker: str) -> None:
    """
    Pour chaque fichier SEC_RAW :
      1. Calcule les croissances YoY pour TOUTES les métriques présentes.
      2. Calcule les ratios financiers + leur croissance YoY.
      3. Ajoute le prix pré-filing (j-1 ou avant) + les prix futurs à
         +1j, +1s, +2s, +1m, +2m, +3m + les rendements associés.
      4. Sauvegarde dans SEC_FINAL.
    """
    print(f"  📊 {ticker}")

    df = pd.read_csv(input_path)
    if len(df) < 2:
        print(f"    ⚠️  Pas assez de lignes ({len(df)}) — ignoré")
        return

    df["filed"] = pd.to_datetime(df["filed"])
    df = df.sort_values("filed")

    # Colonnes métriques brutes = tout sauf metadata
    metric_cols = [c for c in df.columns if c not in META_COLS]

    # ── Croissances + ratios par période (Q1/Q2/Q3/K10) ─────────────────
    results = []

    for period, df_p in df.groupby("period", sort=False):
        df_p    = df_p.sort_values("filed").reset_index(drop=True)
        df_prev = df_p.shift(1)

        for i in range(1, len(df_p)):
            curr = df_p.iloc[i]
            prev = df_prev.iloc[i]

            if pd.isna(prev["year"]):
                continue

            row_out = {
                "filed":         curr["filed"],
                "year_current":  curr["year"],
                "year_previous": prev["year"],
                "period":        period,
            }

            # ── 1. Growth rates (métriques brutes) ──────────────────────
            for m in metric_cols:
                g = safe_growth(curr.get(m), prev.get(m))
                if not pd.isna(g):
                    row_out[f"{m}_growth"] = g

            # ── 2. Ratios courants ───────────────────────────────────────
            for ratio_name, (num_col, den_col) in RATIOS.items():
                val_curr = safe_ratio(curr.get(num_col), curr.get(den_col))
                row_out[ratio_name] = val_curr

            # ── 3. Growth des ratios (ratio_t vs ratio_t-1) ─────────────
            for ratio_name, (num_col, den_col) in RATIOS.items():
                val_curr = safe_ratio(curr.get(num_col), curr.get(den_col))
                val_prev = safe_ratio(prev.get(num_col), prev.get(den_col))
                g_ratio  = safe_growth(val_curr, val_prev)
                if not pd.isna(g_ratio):
                    row_out[f"{ratio_name}_growth"] = g_ratio

            if len(row_out) > 4:
                results.append(row_out)

    if not results:
        print(f"    ⚠️  Aucun résultat — ignoré")
        return

    df_out = pd.DataFrame(results).sort_values("filed")

    # ── Téléchargement prix (une seule fois par ticker) ──────────────────
    df_price = download_prices(ticker)
    if df_price is None:
        print(f"    ❌ Pas de prix Yahoo — colonnes prix seront NaN")

    # ── Collecte des prix & rendements ──────────────────────────────────
    prices_now                              = []
    future_prices  = {label: [] for label, _ in PRICE_HORIZONS}
    future_returns = {label: [] for label, _ in PRICE_HORIZONS}

    for filed_date in df_out["filed"]:

        # ── Prix au filing : dernier cours AVANT le jour J ───────────────
        # (le jour de publication des résultats est exclu)
        p_now = None if df_price is None else nearest_price_before(df_price, filed_date)
        prices_now.append(p_now if p_now is not None else np.nan)

        # ── Prix futurs + rendements ─────────────────────────────────────
        for label, offset in PRICE_HORIZONS:
            if df_price is None or p_now is None:
                future_prices[label].append(np.nan)
                future_returns[label].append(np.nan)
                continue

            target_date = filed_date + offset
            p_future    = nearest_price_after(df_price, target_date)

            future_prices[label].append(p_future if p_future is not None else np.nan)
            future_returns[label].append(
                (p_future - p_now) / p_now if p_future is not None else np.nan
            )

    # ── Ajout des colonnes prix & rendements dans le DataFrame ──────────
    df_out["price_t"] = prices_now

    for label, _ in PRICE_HORIZONS:
        df_out[f"price_t_plus_{label}"] = future_prices[label]
        df_out[f"return_{label}"]       = future_returns[label]

    # Supprime les lignes où le prix pré-filing est manquant
    df_out = df_out.dropna(subset=["price_t"])

    if df_out.empty:
        print(f"    ⚠️  Aucune ligne valide après ajout des prix — ignoré")
        return

    # ── Sauvegarde ───────────────────────────────────────────────────────
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    df_out.to_csv(output_path, index=False)
    print(f"    ✅ {len(df_out)} lignes — {df_out.shape[1]} colonnes")


# ──────────────────────────────────────────
# BOUCLE PRINCIPALE
# ──────────────────────────────────────────

failed = []

for root, _, files in os.walk(RAW_DIR):
    for file in sorted(files):
        if not file.endswith(".csv"):
            continue

        ticker      = file.replace(".csv", "")
        input_path  = os.path.join(root, file)
        rel         = os.path.relpath(root, RAW_DIR)
        output_path = os.path.join(FINAL_DIR, rel, file)

        try:
            compute_growth_and_prices(input_path, output_path, ticker)
            time.sleep(0.2)
        except Exception as e:
            print(f"  ❌ ERREUR {ticker}: {e}")
            failed.append(ticker)

print(f"\n✅ DONE — {len(failed)} échec(s) : {failed}")

  AA
    ❌ Erreur prix AA: Expecting value: line 1 column 1 (char 0)
    Pas de prix Yahoo — colonnes prix seront NaN
     Aucune ligne valide après ajout des prix — ignoré
  ACNT
    ❌ Erreur prix ACNT: Expecting value: line 1 column 1 (char 0)
    Pas de prix Yahoo — colonnes prix seront NaN
     Aucune ligne valide après ajout des prix — ignoré
  ALB-PA
    ❌ Erreur prix ALB-PA: Expecting value: line 1 column 1 (char 0)
    Pas de prix Yahoo — colonnes prix seront NaN
     Aucune ligne valide après ajout des prix — ignoré
  ALB
    ❌ Erreur prix ALB: Expecting value: line 1 column 1 (char 0)
    Pas de prix Yahoo — colonnes prix seront NaN
     Aucune ligne valide après ajout des prix — ignoré
  ALOY
    ❌ Erreur prix ALOY: Expecting value: line 1 column 1 (char 0)
    Pas de prix Yahoo — colonnes prix seront NaN
     Aucune ligne valide après ajout des prix — ignoré
  ALTO
    ❌ Erreur prix ALTO: Expecting value: line 1 column 1 (char 0)
    Pas de prix Yahoo — colonnes prix seront

  AA
    ❌ Erreur prix AA: Expecting value: line 1 column 1 (char 0)
    Pas de prix Yahoo — colonnes prix seront NaN
     Aucune ligne valide après ajout des prix — ignoré
  ACNT
    ❌ Erreur prix ACNT: Expecting value: line 1 column 1 (char 0)
    Pas de prix Yahoo — colonnes prix seront NaN
     Aucune ligne valide après ajout des prix — ignoré
  ALB-PA
    ❌ Erreur prix ALB-PA: Expecting value: line 1 column 1 (char 0)
    Pas de prix Yahoo — colonnes prix seront NaN
     Aucune ligne valide après ajout des prix — ignoré
  ALB
    ❌ Erreur prix ALB: Expecting value: line 1 column 1 (char 0)
    Pas de prix Yahoo — colonnes prix seront NaN
     Aucune ligne valide après ajout des prix — ignoré
  ALOY
    ❌ Erreur prix ALOY: Expecting value: line 1 column 1 (char 0)
    Pas de prix Yahoo — colonnes prix seront NaN
     Aucune ligne valide après ajout des prix — ignoré
  ALTO
    ❌ Erreur prix ALTO: Expecting value: line 1 column 1 (char 0)
    Pas de prix Yahoo — colonnes prix seront

KeyboardInterrupt: 

## Résumé final

In [ ]:
summary = []

for folder in sorted(os.listdir(FINAL_DIR)):
    path = os.path.join(FINAL_DIR, folder)
    if not os.path.isdir(path):
        continue
    files = [f for f in os.listdir(path) if f.endswith(".csv")]
    n_rows = 0
    for f in files:
        try:
            n_rows += len(pd.read_csv(os.path.join(path, f)))
        except Exception:
            pass
    summary.append({"Sector": folder, "Fichiers": len(files), "Lignes totales": n_rows})

df_summary = pd.DataFrame(summary)
print(df_summary.to_string(index=False))
print(f"\nTotal fichiers  : {df_summary['Fichiers'].sum()}")
print(f"Total lignes    : {df_summary['Lignes totales'].sum()}")